In [40]:
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict
import pybedtools

In [41]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
HIMORNA_INITIAL = DATA_INITIAL / "himorna"
HIMORNA = DATA / "himorna"
HM_INITIAL = HIMORNA_INITIAL / "histone_modifications"
HM = HIMORNA / "histone_modifications"
METHYL_INITIAL = HIMORNA_INITIAL / "methylation"
METHYL = HIMORNA / "methylation"
FANTOM = DATA / "fantom_cat"
PREPROCESSING = Path("../preprocessing")
HIMORNA_INDEX = PREPROCESSING / "himorna_index"
HIMORNA_INDEX_V2 = PREPROCESSING / "himorna_index-v2"
HIMORNA_GENES_SIGN =  PREPROCESSING / "himorna_genes_sign"
HIMORNA_GENES_SIGN_V2 =  PREPROCESSING / "himorna_genes_sign-v2"
HIMORNA_INDEX_BED = PREPROCESSING / "himorna_index-bed"
HIMORNA_INDEX_BED_V2 = PREPROCESSING / "himorna_index-bed-v2"

In [4]:
to_himorna_peaks = HIMORNA / "himorna_peaks-hm_meth-assoc_with_names.tsv.gz"
himorna_peaks_assoc = pd.read_csv(
        to_himorna_peaks,
        chunksize=100000000,
        sep="\t",
        index_col=False,
        usecols=["hm","new-lnc","new-lnc_ens","peak","chrom","start","end","corr","new-gene_name","new-ens_gene"]
    )
himorna_peaks_assoc = pd.concat(himorna_peaks_assoc)
print("himorna_peaks_assoc: done")

himorna_peaks_assoc: done


In [5]:
himorna_peaks_assoc.head()

,hm,new-lnc,new-lnc_ens,peak,chrom,start,end,corr,new-gene_name,new-ens_gene
0,H3K27ac,ENSG00000279885,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356,NaN,NaN
1,H3K27ac,ENSG00000277135,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105,NaN,NaN
2,H3K27ac,ENSG00000273133,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511,NaN,NaN
3,H3K27ac,ENSG00000151303,ENSG00000151303,peak_389211,chr18,10441312,10441530,0.491645,NaN,NaN
4,H3K27ac,ENSG00000267707,ENSG00000267707,peak_389211,chr18,10441312,10441530,0.473911,NaN,NaN


In [95]:
himorna_peaks_assoc[(himorna_peaks_assoc["new-lnc"] == "EMX2OS") & (himorna_peaks_assoc["hm"] == "H3K9ac")]

,hm,new-lnc,new-lnc_ens,peak,chrom,start,end,corr,new-gene_name,new-ens_gene


In [6]:
himorna_peaks_assoc.shape

(836906504, 10)

In [74]:
mask = himorna_peaks_assoc["new-lnc_ens"].apply(lambda x: "retired" not in x)
himorna_peaks_assoc_without_retired = himorna_peaks_assoc[mask]
himorna_peaks_assoc_without_retired.head(3)

,hm,new-lnc,new-lnc_ens,peak,chrom,start,end,corr,new-gene_name,new-ens_gene
0,H3K27ac,ENSG00000279885,ENSG00000279885,peak_389211,chr18,10441312,10441530,0.500356,NaN,NaN
1,H3K27ac,ENSG00000277135,ENSG00000277135,peak_389211,chr18,10441312,10441530,0.480105,NaN,NaN
2,H3K27ac,ENSG00000273133,ENSG00000273133,peak_389211,chr18,10441312,10441530,0.450511,NaN,NaN


In [75]:
himorna_peaks_assoc_without_retired.shape

(811542454, 10)

In [76]:
himorna_peaks_assoc_without_retired = himorna_peaks_assoc_without_retired.drop_duplicates()
himorna_peaks_assoc_without_retired.shape

IOStream.flush timed out


(673381914, 10)

In [77]:
mask = himorna_peaks_assoc_without_retired["hm"].isna()
himorna_peaks_assoc_without_retired[mask]

,hm,new-lnc,new-lnc_ens,peak,chrom,start,end,corr,new-gene_name,new-ens_gene


In [78]:
mask = himorna_peaks_assoc_without_retired["new-lnc_ens"].isna()
himorna_peaks_assoc_without_retired[mask]

,hm,new-lnc,new-lnc_ens,peak,chrom,start,end,corr,new-gene_name,new-ens_gene


In [79]:
himorna_peaks_assoc_without_retired = himorna_peaks_assoc_without_retired.dropna(subset=["hm","new-lnc_ens"])
himorna_peaks_assoc_without_retired.shape

(673381914, 10)

In [12]:
himorna_peaks_assoc_without_retired_bed = himorna_peaks_assoc_without_retired[["hm", "new-lnc_ens", "chrom", "start", "end"]] 

In [13]:
himorna_peaks_assoc_without_retired_bed = himorna_peaks_assoc_without_retired_bed.drop_duplicates()

In [14]:
himorna_peaks_assoc_without_retired_bed.shape

(503309958, 5)

In [42]:
def indexing_himorna_by_bed(hm, lnc_ens, subdf):
    to_index = HIMORNA_INDEX_BED_V2 / hm
    to_index.mkdir(exist_ok=True, parents=True)
    to_save = to_index / f"{hm}-{lnc_ens}-sort.bed"
    if to_save.exists():
        print(f"Skipped: {hm} {lnc_ens}")
        return None
    subdf_bed = pybedtools.BedTool.from_dataframe(subdf)
    subdf_bed_sortbed = subdf_bed.sort()
    subdf_bed_sortbed.moveto(to_save)

In [44]:
with Parallel(n_jobs=-1) as pool:
    chunks = pool(
        delayed(indexing_himorna_by_bed)(hm, lnc_ens, subdf[["chrom", "start", "end"]])
        for (hm, lnc_ens), subdf in himorna_peaks_assoc_without_retired_bed.groupby(by=["hm", "new-lnc_ens"])
    )

In [80]:
himorna_peaks_assoc_without_retired["corr_sign"] = np.where(himorna_peaks_assoc_without_retired["corr"] > 0, "plus", "minus")

In [81]:
himorna_peaks_assoc_without_retired = himorna_peaks_assoc_without_retired.dropna(subset=["hm","new-lnc_ens","new-ens_gene"])
himorna_peaks_assoc_without_retired.shape

(534351899, 11)

In [82]:
himorna_peaks_assoc_without_retired[["hm", "new-lnc", "new-lnc_ens", "chrom", "new-gene_name", "new-ens_gene"]] = himorna_peaks_assoc_without_retired[["hm", "new-lnc", "new-lnc_ens", "chrom", "new-gene_name", "new-ens_gene"]].astype('str')

In [83]:
himorna_peaks_assoc_without_retired[["start", "end"]] = himorna_peaks_assoc_without_retired[["start", "end"]].astype(np.uint32)

In [84]:
himorna_peaks_assoc_without_retired

,hm,new-lnc,new-lnc_ens,peak,chrom,start,end,corr,new-gene_name,new-ens_gene,corr_sign
36,H3K27ac,ENSG00000280018,ENSG00000280018,peak_389218,chr18,10446693,10447377,0.579268,CATG00000036918,CATG00000036918,plus
37,H3K27ac,LINC01224,ENSG00000269416,peak_389218,chr18,10446693,10447377,0.410813,CATG00000036918,CATG00000036918,plus
38,H3K27ac,GLYCTK-AS1,ENSG00000242797,peak_389218,chr18,10446693,10447377,0.419894,CATG00000036918,CATG00000036918,plus
39,H3K27ac,ERVH48-1,ENSG00000233056,peak_389218,chr18,10446693,10447377,0.529785,CATG00000036918,CATG00000036918,plus
40,H3K27ac,ENSG00000279148,ENSG00000279148,peak_389219,chr18,10448004,10448179,-0.524631,CATG00000036918,CATG00000036918,minus
...,...,...,...,...,...,...,...,...,...,...,...
836906499,methylation,CD300LD-AS1,ENSG00000182352,peak_47069123,chr8,9741473,9741474,-0.890000,TNKS,ENSG00000173273,minus
836906500,methylation,CD300LD-AS1,ENSG00000182352,peak_47069123,chr8,9741473,9741474,-0.890000,MIR597,ENSG00000207701,minus
836906501,methylation,CD300LD-AS1,ENSG00000182352,peak_47097647,chr8,10827627,10827628,-0.879000,SOX7,ENSG00000171056,minus
836906502,methylation,CD300LD-AS1,ENSG00000182352,peak_47097647,chr8,10827627,10827628,-0.879000,PINX1,ENSG00000258724,minus


In [30]:
def indexing_himorna(hm, lnc_ens, corr_sign, subdf):
    to_index = HIMORNA_INDEX_V2 / hm
    to_index.mkdir(exist_ok=True, parents=True)
    to_save = to_index / f"{hm}-{lnc_ens}-{corr_sign}-.tsv.gz"
    if to_save.exists():
        print(f"Skipped: {hm} {lnc_ens} {corr_sign}")
        return None
    subdf.to_csv(to_save, sep="\t", compression="gzip", index=False)

In [34]:
with Parallel(n_jobs=-1) as pool:
    chunks = pool(
        delayed(indexing_himorna)(hm, lnc_ens, corr_sign, subdf)
        for (hm, lnc_ens, corr_sign), subdf in himorna_peaks_assoc_without_retired.groupby(by=["hm", "new-lnc_ens", "corr_sign"])
    )

In [9]:
H3K36me3_ENSG00000267110 = pd.concat([pd.read_csv(to_file, sep="\t") for to_file in HIMORNA_INDEX.glob("H3K36me3/*ENSG00000267110*")])
H3K36me3_ENSG00000267110

ValueError: No objects to concatenate

In [35]:
def indexing_himorna_genes(hm, lnc_ens):
    """
    Это верный способ извлекать plus и minus скоррелирвоанные гены. Посколько наивный подход не учитывает то, 
    что один и тот же ген может иметь как plus, так и minus пики. Данная функция приписывает знак гену по принципу "Тот знак, где больше пиков"
    """
    df = pd.concat([pd.read_csv(i, sep="\t") for i in HIMORNA_INDEX_V2.glob(f'{hm}/{hm}-{lnc_ens}-*-.tsv.gz')])
    genes_ens_dict = defaultdict(list)
    for gene_ens, subdf in df.groupby(by=["new-ens_gene"]):
        corr_sign = subdf.value_counts(subset=["corr_sign"]).idxmax()[0]
        genes_ens_dict[corr_sign].append(gene_ens[0])
    
    for corr_sign in genes_ens_dict:
        print(corr_sign)
        to_index = HIMORNA_GENES_SIGN_V2 / hm
        to_index.mkdir(exist_ok=True, parents=True)
        to_save = to_index / f"{hm}-{lnc_ens}-{corr_sign}-genes_list-.tsv.gz"
        pd.DataFrame({"genes_list":genes_ens_dict[corr_sign]}).to_csv(to_save, sep="\t", compression="gzip", index=False)

In [36]:
list_of_hm = himorna_peaks_assoc_without_retired["hm"].unique()

In [37]:
list_of_hm

array(['H3K27ac', 'H3K27me3', 'H3K36me3', 'H3K4me1', 'H3K4me2', 'H3K4me3',
       'H3K79me2', 'H3K9ac', 'H3K9me3', 'H4K20me1', 'methylation'],
      dtype=object)

In [38]:
with Parallel(n_jobs=-1) as pool:
    chunks = pool(
        delayed(indexing_himorna_genes)(hm, lnc_ens)
        for hm in list_of_hm for lnc_ens in [i.stem.split("-")[1] for i in HIMORNA_INDEX_V2.glob(f'{hm}/{hm}-*-.tsv.gz')]
    )

minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
plus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
minus
plus
plus
minus
plus
minus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
plus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
minus
minus
plus
minus
plus
minus
plus
plus
minus
plus
plus
minus
minus
plus
plus
minus
minus
plus
minus
plus
plus
minus
plus
minus
plus
plus
minus
minus
minus
plus
plus
minus
plus
minus
minus
plus
minus
plus
plus
plus
minus
plus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
plus
minus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
minus
plus
minus
minus
plus
plus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
plus
minus
plus
minus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
plus
minus
plus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus


In [39]:
to_himorna = HIMORNA_GENES_SIGN_V2 / "H3K27ac/H3K27ac-ENSG00000223813-plus-genes_list-.tsv.gz"
plus_genes_H3K27ac_ENSG00000223813 = pd.read_csv(to_himorna, sep="\t")
plus_genes_H3K27ac_ENSG00000223813

,genes_list
0,CATG00000000117
1,CATG00000000173
2,CATG00000000331
3,CATG00000000490
4,CATG00000000496
...,...
2804,ENSG00000273448
2805,ENSG00000283982
2806,ENSG00000284634
2807,ENSG00000285567


In [54]:
to_himorna = HIMORNA_GENES_SIGN / "H3K27ac/H3K27ac-ENSG00000223813-plus-genes_list-.tsv.gz"
plus_genes_H3K27ac_ENSG00000223813 = pd.read_csv(to_himorna, sep="\t")
plus_genes_H3K27ac_ENSG00000223813

,genes_list
0,CATG00000000117
1,CATG00000000173
2,CATG00000000331
3,CATG00000000490
4,CATG00000000496
...,...
2878,ENSG00000272848
2879,ENSG00000272865
2880,ENSG00000272917
2881,ENSG00000273274


plus
minus
minus
plus
minus
plus
plus
minus
plus
plus
minus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
plus
minus
plus
minus
plus
plus
plus
minus
plus
plus
minus
minus
plus
minus
plus
plus
plus
plus
plus
minus
minus
plus
minus
plus
minus
plus
plus
minus
plus
minus
minus
minus
plus
minus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
minus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
minus
plus
minus
plus
minus
minus
plus
minus
plus
minus
plus
plus
minus
minus
minus
plus
plus
minus
minus
plus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
plus
minus
minus
plus
minus
plus
plus
minus
minus
plus
plus
minus
plus
minus
minus
plus
plus
minus
minus
plus
minus
plus
plus
plus
minus
plus
minus
minus
plus
plus
minus
plus
minus
plus
minus
minus
plus


In [ ]:
# проверка общего количество генов

In [71]:
to_file = FANTOM / 'fantom_l2-hg19-sort.bed'
fantom_hg19 = pd.read_csv(to_file, sep="\t",  names=["chrom","start","end", "name"], usecols=[0,1,2,3], header=0)
mask = fantom_hg19["name"].apply(lambda x: "gene" in x)
fantom_genes_hg19 = fantom_hg19[mask]

In [72]:
to_file = FANTOM / 'fantom_l2-genes_with_name-hg38-all_info.tsv'
fantom_l2_genes_update_with_names = pd.read_csv(to_file, sep="\t")
mask = fantom_l2_genes_update_with_names["new-ens_gene"].apply(lambda x: "retired" not in x)
fantom_l2_genes_update_with_names_without_retired = fantom_l2_genes_update_with_names[mask]

In [85]:
himorna_peaks_assoc_without_retired

,hm,new-lnc,new-lnc_ens,peak,chrom,start,end,corr,new-gene_name,new-ens_gene,corr_sign
36,H3K27ac,ENSG00000280018,ENSG00000280018,peak_389218,chr18,10446693,10447377,0.579268,CATG00000036918,CATG00000036918,plus
37,H3K27ac,LINC01224,ENSG00000269416,peak_389218,chr18,10446693,10447377,0.410813,CATG00000036918,CATG00000036918,plus
38,H3K27ac,GLYCTK-AS1,ENSG00000242797,peak_389218,chr18,10446693,10447377,0.419894,CATG00000036918,CATG00000036918,plus
39,H3K27ac,ERVH48-1,ENSG00000233056,peak_389218,chr18,10446693,10447377,0.529785,CATG00000036918,CATG00000036918,plus
40,H3K27ac,ENSG00000279148,ENSG00000279148,peak_389219,chr18,10448004,10448179,-0.524631,CATG00000036918,CATG00000036918,minus
...,...,...,...,...,...,...,...,...,...,...,...
836906499,methylation,CD300LD-AS1,ENSG00000182352,peak_47069123,chr8,9741473,9741474,-0.890000,TNKS,ENSG00000173273,minus
836906500,methylation,CD300LD-AS1,ENSG00000182352,peak_47069123,chr8,9741473,9741474,-0.890000,MIR597,ENSG00000207701,minus
836906501,methylation,CD300LD-AS1,ENSG00000182352,peak_47097647,chr8,10827627,10827628,-0.879000,SOX7,ENSG00000171056,minus
836906502,methylation,CD300LD-AS1,ENSG00000182352,peak_47097647,chr8,10827627,10827628,-0.879000,PINX1,ENSG00000258724,minus


In [89]:
himorna_peaks_assoc_without_retired_genes = himorna_peaks_assoc_without_retired["new-ens_gene"].unique()
fantom_genes_hg19_genes = fantom_genes_hg19["name"].unique()
fantom_genes_hg38_genes = fantom_l2_genes_update_with_names_without_retired["new-ens_gene"].unique()

In [90]:
len(himorna_peaks_assoc_without_retired_genes), len(fantom_genes_hg19_genes), len(fantom_genes_hg38_genes)

(121361, 124245, 121447)